In [1]:
import pandas as pd
import os

code_analysis_generated_path = "../../generated/code-analysis"
if not os.path.exists(code_analysis_generated_path):
    os.makedirs(code_analysis_generated_path)

reports_path = "../../generated/reports-no-bug"

data = []

users = os.listdir(reports_path)
if len(users) == 0:
    raise FileNotFoundError(f"Le dossier {reports_path} est vide. Copier les uploads dans ce dossier avant d'exécuter le notebook.")

for user in users:
    user_path = os.path.join(reports_path, user)
    jacoco_path = os.path.join(user_path, "JaCoCo", "jacoco.csv")
    pitest_path = os.path.join(user_path, "Pitest")
    jacoco_valid = os.path.exists(jacoco_path)
    data.append([user, jacoco_path, jacoco_valid])

main_df = pd.DataFrame(data, columns=['user', 'jacoco', 'jacoco_valid'])


# Add group

In [2]:
users_csv_path = '../../generated/database/users.csv'
if not os.path.exists(users_csv_path):
    raise FileNotFoundError(f"Le fichier {users_csv_path} n'existe pas. Exécutez le notebook 'notebooks/arrange data/Database data.ipynb' pour le générer.")

df_users = pd.read_csv(users_csv_path, usecols=['user', 'game_mode']).dropna()
main_df = main_df.merge(df_users, on='user', how='left')

In [3]:
main_df

,user,jacoco,jacoco_valid,game_mode
0,0947a22a-ec40-4f49-867b-330a623a1a35,../../generated/reports-no-bug\0947a22a-ec40-4...,True,TEAM
1,0d271530-be17-4538-bf04-dde3c6069b5f,../../generated/reports-no-bug\0d271530-be17-4...,True,SOLO
2,11555248-3f01-4d1c-9a71-ef7caf9150fa,../../generated/reports-no-bug\11555248-3f01-4...,True,SOLO
3,70c71a91-06a1-4bf4-8e56-f4327b1c8b3f,../../generated/reports-no-bug\70c71a91-06a1-4...,True,SOLO
4,70e85b5e-92d4-498a-9079-ba881f5b3b82,../../generated/reports-no-bug\70e85b5e-92d4-4...,True,TEAM
5,7234f6dc-a316-4576-8453-6e10d7cf1c3d,../../generated/reports-no-bug\7234f6dc-a316-4...,True,SOLO
6,84c2c4e6-1c27-4f2c-808b-9e84a7cb257e,../../generated/reports-no-bug\84c2c4e6-1c27-4...,True,SOLO
7,960ad9f8-2b83-4526-9cfb-1a59cec049a6,../../generated/reports-no-bug\960ad9f8-2b83-4...,True,SOLO
8,bcec21aa-2246-4ee8-9f2a-3998a5cde697,../../generated/reports-no-bug\bcec21aa-2246-4...,True,TEAM
9,c27240eb-52c0-4436-aa0f-e7f97a94a725,../../generated/reports-no-bug\c27240eb-52c0-4...,True,TEAM


# Get data where JaCoCo report is valid


In [4]:
cols = ['instruction', 'branch', 'line', 'method']
users_with_valid_jacoco = main_df[(main_df['jacoco_valid'] == True)]
jacoco_users = users_with_valid_jacoco[users_with_valid_jacoco['game_mode'].notna()]
print(f"Nombre d'utilisateurs avec un rapports JaCoCo valide : {len(jacoco_users)}")

Nombre d'utilisateurs avec un rapports JaCoCo valide : 13


In [5]:
jacoco_users

,user,jacoco,jacoco_valid,game_mode
0,0947a22a-ec40-4f49-867b-330a623a1a35,../../generated/reports-no-bug\0947a22a-ec40-4...,True,TEAM
1,0d271530-be17-4538-bf04-dde3c6069b5f,../../generated/reports-no-bug\0d271530-be17-4...,True,SOLO
2,11555248-3f01-4d1c-9a71-ef7caf9150fa,../../generated/reports-no-bug\11555248-3f01-4...,True,SOLO
3,70c71a91-06a1-4bf4-8e56-f4327b1c8b3f,../../generated/reports-no-bug\70c71a91-06a1-4...,True,SOLO
4,70e85b5e-92d4-498a-9079-ba881f5b3b82,../../generated/reports-no-bug\70e85b5e-92d4-4...,True,TEAM
5,7234f6dc-a316-4576-8453-6e10d7cf1c3d,../../generated/reports-no-bug\7234f6dc-a316-4...,True,SOLO
6,84c2c4e6-1c27-4f2c-808b-9e84a7cb257e,../../generated/reports-no-bug\84c2c4e6-1c27-4...,True,SOLO
7,960ad9f8-2b83-4526-9cfb-1a59cec049a6,../../generated/reports-no-bug\960ad9f8-2b83-4...,True,SOLO
8,bcec21aa-2246-4ee8-9f2a-3998a5cde697,../../generated/reports-no-bug\bcec21aa-2246-4...,True,TEAM
9,c27240eb-52c0-4436-aa0f-e7f97a94a725,../../generated/reports-no-bug\c27240eb-52c0-4...,True,TEAM


In [6]:
users_jacoco_data = []

for _, user_data in jacoco_users.iterrows():
    csv_file = pd.read_csv(user_data['jacoco']).drop(columns=['GROUP', 'PACKAGE', 'CLASS'])
    data = csv_file.sum().tolist()

    users_jacoco_data.append([user_data['user'], user_data['game_mode']] + data)

df_jacoco = pd.DataFrame(
    users_jacoco_data,
    columns=['user', 'game_mode', 'instruction_missed', 'instruction_covered',
             'branch_missed', 'branch_covered', 'line_missed', 'line_covered',
             'complexity_missed', 'complexity_covered', 'method_missed', 'method_covered']
)

for col in cols:
    df_jacoco[f'{col}'] = df_jacoco[f'{col}_covered'] / (df_jacoco[f'{col}_missed'] + df_jacoco[f'{col}_covered'])



df_jacoco.to_csv(f"{code_analysis_generated_path}/jacoco.csv", index=False)
df_jacoco


,user,game_mode,instruction_missed,instruction_covered,branch_missed,branch_covered,line_missed,line_covered,complexity_missed,complexity_covered,method_missed,method_covered,instruction,branch,line,method
0,0947a22a-ec40-4f49-867b-330a623a1a35,TEAM,1829,1248,102,55,426,258,150,127,84,103,0.405590,0.350318,0.377193,0.550802
1,0d271530-be17-4538-bf04-dde3c6069b5f,SOLO,2817,260,143,14,630,54,249,28,167,20,0.084498,0.089172,0.078947,0.106952
2,11555248-3f01-4d1c-9a71-ef7caf9150fa,SOLO,2782,295,148,9,626,58,258,19,168,19,0.095873,0.057325,0.084795,0.101604
3,70c71a91-06a1-4bf4-8e56-f4327b1c8b3f,SOLO,3005,72,157,0,666,18,269,8,179,8,0.023399,0.000000,0.026316,0.042781
4,70e85b5e-92d4-498a-9079-ba881f5b3b82,TEAM,2536,541,143,14,565,119,234,43,145,42,0.175821,0.089172,0.173977,0.224599
5,7234f6dc-a316-4576-8453-6e10d7cf1c3d,SOLO,2358,719,126,31,538,146,208,69,132,55,0.233669,0.197452,0.213450,0.294118
6,84c2c4e6-1c27-4f2c-808b-9e84a7cb257e,SOLO,2551,526,135,22,569,115,228,49,147,40,0.170946,0.140127,0.168129,0.213904
7,960ad9f8-2b83-4526-9cfb-1a59cec049a6,SOLO,2518,559,139,18,556,128,240,37,157,30,0.181670,0.114650,0.187135,0.160428
8,bcec21aa-2246-4ee8-9f2a-3998a5cde697,TEAM,2927,150,149,8,645,39,259,18,172,15,0.048749,0.050955,0.057018,0.080214
9,c27240eb-52c0-4436-aa0f-e7f97a94a725,TEAM,2170,907,127,30,512,172,206,71,125,62,0.294768,0.191083,0.251462,0.331551
